> **Copia adaptada (aprendizaje/leccion-04).** El original usa Azure OpenAI con la Responses API.
> Esta versión usa **Chat Completions** con **Gemini** (free tier) u **Ollama** (local), según `PROVIDER` en la celda del Ejercicio 2.
> Solo cambian `base_url`, `api_key` y `model`: el resto del código es igual para los dos.

The following notebook was auto-generated by GitHub Copilot Chat and is meant for initial setup only

# Introduction to Prompt Engineering
Prompt engineering is the process of designing and optimizing prompts for natural language processing tasks. It involves selecting the right prompts, tuning their parameters, and evaluating their performance. Prompt engineering is crucial for achieving high accuracy and efficiency in NLP models. In this section, we will explore the basics of prompt engineering using the OpenAI models for exploration.

### Exercise 1: Tokenization
Explore Tokenization using tiktoken, an open-source fast tokenizer from OpenAI
See [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) for more examples.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
# NOTA: tiktoken es el tokenizer de OpenAI. Gemini y Llama tokenizan distinto:
# sirve para la intuición, pero el conteo no es exacto para esos modelos.
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### Exercise 2: Validate OpenAI API Key Setup

Run the code below to verify that your OpenAI endpoint is set up correctly. The code just tries a simple basic prompt and validates the completion. Input `oh say can you see` should complete along the lines of `by the dawn's early light..`


In [ ]:
# Adaptado: Chat Completions con Gemini u Ollama (ambos compatibles con la API de OpenAI)

import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

PROVIDER = "ollama"   # cambia a "ollama" para comparar con el modelo local

PROVEEDORES = {
    "gemini": {
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
        "api_key": os.environ.get("GEMINI_API_KEY"),
        "model": os.environ.get("GEMINI_MODEL"),
    },
    "ollama": {
        "base_url": os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
        "api_key": "ollama",  # Ollama no valida la key, pero el cliente exige una
        "model": os.environ.get("OLLAMA_MODEL", "llama3.2:3b"),
    },
}
cfg = PROVEEDORES[PROVIDER]

# max_retries: reintenta ante 429/503 (saturación temporal del free tier)
client = OpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"], max_retries=5)
deployment = cfg["model"]

def get_completion(prompt, as_json=False):
    params = {
        "model": deployment,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1024,
    }
    
    if as_json:
        params["response_format"] = {"type": "json_object"}
        
    response = client.chat.completions.create(**params)
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
You should respond with a title and a short summary of the text above. The title should be 5 words or less, and the summary should be 20 words or less.
Also, you should respond in the next format JSON:{{"title" : <title>, "summary" : <summary>}} and nothing else. Do not include any other text or explanation.
"""

## 3. Run the prompt
print(f"[{PROVIDER} · {deployment}]")
response = get_completion(prompt, as_json=True)
print(response)


In [ ]:
import json
try:
    datos = json.loads(response)
    print("✅ JSON válido:", datos["title"])
except json.JSONDecodeError as err:
    print("❌ JSON inválido:", err)


#### 📘 Referencia AI-103: cómo sería en Azure OpenAI (no ejecutar: sin créditos de Azure)

**Opción A — Responses API** (la que usa el notebook original; la más nueva):
```python
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",  # endpoint v1: sin api_version
)
response = client.responses.create(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],   # nombre del DEPLOYMENT, no del modelo
    input=prompt,                                  # string, o lista de mensajes con role/content
    max_output_tokens=1024,
    store=False,                                   # no guardar la respuesta en el servicio
)
print(response.output_text)
```

**Opción B — Chat Completions con el cliente `AzureOpenAI`** (la de mi prueba de humo de la lección 00):
```python
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2024-10-21",                      # obligatorio con este cliente
)
response = client.chat.completions.create(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    messages=[{"role": "user", "content": prompt}],
    max_tokens=1024,                               # en modelos de razonamiento: max_completion_tokens
)
print(response.choices[0].message.content)
```

| | Responses API | Chat Completions |
|---|---|---|
| Método | `client.responses.create(...)` | `client.chat.completions.create(...)` |
| Entrada | `input=` (string o lista de mensajes) | `messages=` (lista de mensajes) |
| Límite de salida | `max_output_tokens=` | `max_tokens=` / `max_completion_tokens=` |
| Leer la respuesta | `response.output_text` | `response.choices[0].message.content` |
| Cliente en Azure | `OpenAI(base_url=.../openai/v1/)` | `AzureOpenAI(azure_endpoint=..., api_version=...)` |
| Estado en el servidor | Puede guardar la conversación (`store`, `previous_response_id`) | Sin estado: reenvías el historial completo |


#### 📘 Referencia AI-103: structured outputs en Azure OpenAI (no ejecutar)

**Chat Completions** (`AzureOpenAI` o `OpenAI` con `/openai/v1/`):
```python
response = client.chat.completions.create(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    messages=[{"role": "user", "content": prompt}],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "resumen",
            "strict": True,                      # obliga a respetar el esquema
            "schema": {
                "type": "object",
                "properties": {
                    "title":   {"type": "string"},
                    "summary": {"type": "string"},
                },
                "required": ["title", "summary"],
                "additionalProperties": False,
            },
        },
    },
)
datos = json.loads(response.choices[0].message.content)
```

**Responses API**: el esquema va en `text.format` (no en `response_format`):
```python
response = client.responses.create(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    input=prompt,
    text={"format": {"type": "json_schema", "name": "resumen", "strict": True, "schema": {...}}},
)
datos = json.loads(response.output_text)
```

| Nivel | Qué garantiza |
|---|---|
| Formato pedido en el prompt | Nada (instrucción probabilística) |
| `{"type": "json_object"}` | JSON válido, con cualquier estructura |
| `{"type": "json_schema", ...}` | JSON válido **con los campos y tipos** del esquema |
| + validar en el código (Pydantic) | Que el backend nunca procese datos con forma inesperada |


### Exercise 3: Fabrications
Explore what happens when you ask the LLM to return completions for a prompt about a topic that may not exist, or about topics that it may not know about because it was outside it's pre-trained dataset (more recent). See how the response changes if you try a different prompt, or a different model.

In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
Haz un plan de clase sobre el Tratado de Córdoba de 1823 entre Argentina y Chile.
"""

prompt = f"""
```{text}```
If you don't know what it is about, do not make it up.
"""

response = get_completion(prompt)
print(response)

### Exercise 4: Instruction Based 
Use the "text" variable to set the primary content 
and the "prompt" variable to provide an instruction related to that primary content.

Here we ask the model to summarize the text for a second-grade student

In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student. Keep the result to one paragraph with 3-5 bullet points.
The summary should be concise and brief. Keep in mind that it is intended for a teacher to use with children around nine years old. Do not invent content; use only the information found in the text. 
```{text}```
Jupiter is the largest planet in the Solar System...
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### Exercise 5: Complex Prompt 
Try a request that has system, user and assistant messages 
System sets assistant context
User & Assistant messages provide multi-turn conversation context

Note how the assistant personality is set to "sarcastic" in the system context. 
Try using a different personality context. Or try a different series of input/output messages

In [ ]:
# Adaptado: Chat Completions (messages) en lugar de Responses API (input)
response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ],
)
print(response.choices[0].message.content)


#### 📘 Referencia AI-103: el Ejercicio 5 en Azure (Responses API, no ejecutar)
```python
response = client.responses.create(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    input=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"},
    ],
    store=False,
)
print(response.output_text)
```
Con Chat Completions + `AzureOpenAI` es igual que la celda de arriba, cambiando solo el cliente (ver referencia del Ejercicio 2).


#### 📘 Referencia AI-103: ¿quién guarda el historial? (no ejecutar)

**Chat Completions — sin estado (*stateless*):** el historial lo guardas **tú** y lo reenvías completo en cada llamada.
```python
historial = [{"role": "system", "content": "You are a sarcastic assistant."}]

historial.append({"role": "user", "content": "Who won the world series in 2020?"})
r1 = client.chat.completions.create(model=deployment, messages=historial)
historial.append({"role": "assistant", "content": r1.choices[0].message.content})  # guardo la respuesta

historial.append({"role": "user", "content": "Where was it played?"})
r2 = client.chat.completions.create(model=deployment, messages=historial)          # reenvío TODO
```

**Responses API — el servidor puede guardar el estado:** la segunda llamada solo manda el mensaje nuevo
y referencia la respuesta anterior por su id. Ejemplo oficial de Microsoft Learn
([Chaining responses](https://learn.microsoft.com/azure/foundry/openai/how-to/responses#chaining-responses-together)):
```python
client = OpenAI(
    base_url="https://YOUR-RESOURCE-NAME.openai.azure.com/openai/v1/",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

first_response = client.responses.create(
    model="MODEL_NAME",                        # nombre del deployment
    input="Define catastrophic forgetting.",
)

second_response = client.responses.create(
    model="MODEL_NAME",
    previous_response_id=first_response.id,    # el servidor recupera el contexto anterior
    input="Explain it for a college freshman.",  # solo el mensaje nuevo
)
print(second_response.output_text)
```

⚠️ **`store=False` y `previous_response_id` no se combinan:** si la primera respuesta se creó con `store=False`
(como en el notebook original), el servidor **no la guardó** y no se puede encadenar.

| | Chat Completions | Responses API |
|---|---|---|
| ¿Quién guarda el historial? | **Tu backend** (lista `messages`) | El **servidor**, si `store` está activo |
| ¿Qué envías en cada turno? | El historial **completo** | Solo el mensaje nuevo + `previous_response_id` |
| Tokens de entrada | Se reenvían y se cobran en cada llamada | El contexto anterior también cuenta como entrada, aunque no lo reenvíes |
| Control y privacidad | Total: decides qué guardar y qué recortar | Los datos quedan en el servicio (revisar retención y cumplimiento) |


In [ ]:
text = f"""
Complete the sentence "Show me an image of red car of make Volvo and..."
"""

## Set the prompt
prompt = f"""
```{text}```
You must respond with a single sentence, incorporating details such as the year, the location, and the number of people inside the car. The goal is for the finished sentence to serve as a clear and precise prompt for generating an image using a model like gpt-image-2.5-flare.
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### Exercise: Explore Your Intuition
The above examples give you patterns that you can use to create new prompts (simple, complex, instruction etc.) - try creating other exercises to explore some of the other ideas we've talked about like examples, cues and more.

In [ ]:
system_prompt = f"""
You are a friendly and helpful barbershop assistant. Service hours are from 9:00 AM to 1:00 PM and from 3:00 PM to 9:00 PM. The services offered are haircuts, haircuts with beard grooming, and facial treatments. Please explain the cancellation policy: cancellations must be made at least 24 hours in advance. Inform the client that failing to cancel within this timeframe counts as a missed appointment, and a second missed appointment will result in them no longer being able to book future slots.
You need to keep the date in mind; when they give you the day, it means from that day onwards.
"""
Assistant_prompt = f"""
Hi, I'm Bartun, the barbershop's AI assistant. Let me check the schedule...
I don't have availability for that time, but I can offer you 5:00 PM or 7:00 PM.
"""

response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Bro, do you have an opening for tomorrow at 6 PM?"},
        {"role": "assistant", "content": Assistant_prompt},
        {"role": "user", "content": "Perfect, 7 PM works for me. Book it."},
    ],
)
print(response.choices[0].message.content)